In [7]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

"""
Metrics this file produces:
1. Macro-F1 vs model (colored by modality)
- Shows which architectures work best
2. Accuracy vs Macro-F1 scatter
- Majority class cheating
3. Delta-from-best per modality
- Performance degradation
4. Frozen vs unfrozen comparison
- Representation learning effectiveness
5. Size comparison bars
- Compression effects
6. Model size vs Macro-F1 line plot
- Central thesis argument (lightweight models sometimes approach performance of larger ones)
7. KL convergence for TinyLlama
- Determines whether model is certain or uncertain about predictions (stability vs spikes in uncertainty)
"""

import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.special import softmax
import pandas as pd
import gc
import torch
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# add project path to sys.path
PROJECT_PATH = "/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Code Files"

sys.path.append(PROJECT_PATH)

# import models
from rf_baseline import run_rf_baseline
from mlp_baseline import run_mlp_baseline
from cnn_lstm import run_cnnlstm
from mae import run_mae
from stmae import run_stmae
from distilbert import run_distilbert
from tinyllama import run_tinyllama

REPEATS = 0

TARGET_REPEAT = 0

RESULTS_ROOT = "/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/results"

CHECKPOINT_FILE = os.path.join(RESULTS_ROOT, "completed_runs.csv")

if os.path.exists(CHECKPOINT_FILE):
    completed_runs = pd.read_csv(CHECKPOINT_FILE)
else:
    completed_runs = pd.DataFrame(columns=["model", "size", "mode", "freeze", "repeat"])

MODE_NAMES = {
    0: "network",
    1: "sensor",
    2: "fusion"
}

MODEL_REGISTRY = {
    0: "rf",
    1: "mlp",
    2: "cnnlstm",
    3: "mae",
    4: "stmae",
    5: "distilbert",
    6: "tinyllama"
}

FREEZE_OPTIONS = {
    'rf': [None],
    'mlp': [None],
    'cnnlstm': [None],
    'mae': [True, False],
    'stmae': [True, False],
    'distilbert': [True, False],
    'tinyllama': [True, False]
}

MODEL_SIZES = {
    'rf': ['base', 'small', 'tiny'],
    'mlp': ['base','small','tiny'],
    'cnnlstm': ['base','small','tiny'],
    'mae': ['base','small', 'tiny'],
    'stmae': ['base', 'small', 'tiny'],
    'distilbert': ['base', 'small', 'tiny'],
    'tinyllama': ['base', 'small', 'tiny']
}

MODEL_FAMILY = {
    'rf': 'classical',
    'mlp': 'classical',
    'cnnlstm': 'sequential-neural',
    'mae': 'representation',
    'stmae': 'representation',
    'distilbert': 'llm',
    'tinyllama': 'llm'
}

BATCH_CONFIG = {
    "tabular": 2048,
    "sequence": 512
}

TARGET_MODELS = ["rf", "cnnlstm", "mae", "tinyllama"]

MODEL_MODE_MAP = {
    "rf": [0],
    "cnnlstm": [0],
    "mae": [2],
    "tinyllama": [2]
}


def create_result_dirs(mode_name):

    os.makedirs(RESULTS_ROOT, exist_ok=True)

    base = os.path.join(RESULTS_ROOT, mode_name)

    os.makedirs(base, exist_ok=True)

    metrics_dir = os.path.join(base, "metrics")
    plots_dir   = os.path.join(base, "plots")

    os.makedirs(metrics_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)

    return metrics_dir, plots_dir


def run_experiment(model_name, mode, freeze_encoder=None, size="base", window=None):
    if model_name == 0:
      return run_rf_baseline(mode, size, batch_size=BATCH_CONFIG["tabular"])

    elif model_name == 1:
      return run_mlp_baseline(mode, size, batch_size=BATCH_CONFIG["tabular"])

    elif model_name == 2:
      return run_cnnlstm(mode, size, window=window, batch_size=BATCH_CONFIG["sequence"])

    elif model_name == 3:
      return run_mae(freeze_encoder, mode, size, batch_size=BATCH_CONFIG["tabular"])

    elif model_name == 4:
      return run_stmae(freeze_encoder, mode, size, window=window, batch_size=BATCH_CONFIG["sequence"])

    elif model_name == 5:
      return run_distilbert(freeze_encoder, mode, size)

    elif model_name == 6:
      return run_tinyllama(freeze_encoder, mode, size, window=window)


def is_completed(model, size, mode, freeze, repeat):
  global completed_runs
  return (
      (completed_runs["model"] == model) &
      (completed_runs["size"] == size) &
      (completed_runs["mode"] == mode) &
      (completed_runs["freeze"] == freeze) &
      (completed_runs["repeat"] == repeat)).any()


def should_skip(model_name, size, freeze):
  if model_name == "distilbert":
    if size == "small":
      return True

  if model_name == "tinyllama":
    if size == "small":
      return True
  return False


def mark_completed(model, size, mode, freeze, repeat):
  global completed_runs

  new_row = {
      "model": model,
      "size": size,
      "mode": mode,
      "freeze": freeze,
      "repeat": repeat,
      "timestamp": pd.Timestamp.now()
  }

  completed_runs = pd.concat([completed_runs, pd.DataFrame([new_row])], ignore_index=True)

  completed_runs.to_csv(CHECKPOINT_FILE, index=False)


def save_metrics(results, tag, metrics_dir):
  filepath = os.path.join(metrics_dir, f"{tag}.txt")
  with open(filepath, "w") as f:
    f.write(f"Model run ID: {tag}\n")
    f.write(f"Accuracy : {results['accuracy']}\n")
    f.write(f"Precision: {results['precision']}\n")
    f.write(f"Recall   : {results['recall']}\n")
    f.write(f"F1 Score : {results['f1']}\n")


def plot_loss(history, save_path):
  plt.figure(figsize=(10, 8), facecolor='lightblue')
  plt.plot(history)
  plt.xlabel("Epoch")
  plt.ylabel("Loss")
  plt.savefig(save_path, dpi=300)
  plt.close()


def plot_confusion(cm, save_path):
  plt.figure(figsize=(10, 8), facecolor='lightblue')
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
  plt.xlabel("Predicted")
  plt.ylabel("Actual")
  plt.savefig(save_path, dpi=300)
  plt.close()


# low KL divergence = good consistency (stable predictions)
# high KL divergence = bad consistency (model uncertainty spikes)
def plot_world_model_consistency(logits, labels, save_path):
  probs = softmax(logits, axis=1)
  kl_vals = []

  for t in range(1, len(probs)):
    p = probs[t]
    q = probs[t-1]

    kl = np.sum(p * np.log((p+1e-9)/(q+1e-9)))
    kl_vals.append(kl)

  plt.figure(figsize=(10, 8), facecolor='lightblue')
  plt.plot(kl_vals)
  plt.xlabel("Time Step")
  plt.ylabel("Prediction Distribution Drift")
  plt.savefig(save_path, dpi=300)
  plt.close()


def load_cm_from_results(model, mode, size="base", encoder="unfrozen"):
  try:
    plots_dir = os.path.join(RESULTS_ROOT, mode, "plots")

    target = f"{model}_{mode}_{size}_{encoder}" if model in ["mae", "stmae", "distilbert", "tinyllama"] else f"{model}_{mode}_{size}"

    for filename in os.listdir(plots_dir):
      if filename.endswith(".npy") and target in filename:
        return np.load(os.path.join(plots_dir, filename))

  except Exception as e:
    print(f"Error loading confusion matrix for {model} in {mode}: {e}")
    return None


def simplify_confusion(cm, top_k=6):
  try:
    class_totals = cm.sum(axis=1)
    top_indices = np.argsort(class_totals)[-top_k:]

    new_cm = cm[np.ix_(top_indices, top_indices)]
    return new_cm
  except Exception as e:
    print(f"Error simplifying confusion matrix: {e}")
    return cm


def plot_normalized_confusion(cm, save_path):
  try:
    cm = cm.astype(float)
    cm /= cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
  except Exception as e:
    print(f"Error plotting normalized confusion matrix: {e}")
    return


def rebuild_summary_from_files():
  rows = []

  for mode in ["network", "sensor", "fusion"]:
    metrics_dir = os.path.join(RESULTS_ROOT, mode, "metrics")

    if not os.path.exists(metrics_dir):
      continue

    for filename in os.listdir(metrics_dir):
      if not filename.endswith(".txt"):
        continue

      path = os.path.join(metrics_dir, filename)

      with open(path, "r") as f:
        lines = f.readlines()

      try:
        model_run_id = filename.replace(".txt", "").split("_")

        model = model_run_id[0]
        mode = model_run_id[1]
        size = model_run_id[2]

        encoder = "none"
        if "unfrozen" in filename:
          encoder = "unfrozen"
        elif "frozen" in filename:
          encoder = "frozen"

        accuracy = float(lines[1].split(":")[1].strip())
        precision = float(lines[2].split(":")[1].strip())
        recall = float(lines[3].split(":")[1].strip())
        f1 = float(lines[4].split(":")[1].strip())

        rows.append({
            "model": model,
            "size": size,
            "mode": mode,
            "encoder": encoder,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

      except Exception as e:
        print(f"Error parsing {path}: {e}")
        print(f"Skipping {path}")
        continue

  df = pd.DataFrame(rows)

  df["family"] = df["model"].map(MODEL_FAMILY)

  return df


# generate summary metrics and charts
def load_summary():
  return pd.read_csv(os.path.join(RESULTS_ROOT, "summary_clean.csv"))


def plot_f1_vs_model(df, save_path):
  sns.set(style="whitegrid")
  plt.figure(figsize=(10, 6))
  sns.barplot(data=df, x="model", y="f1_mean", hue="mode")
  plt.xticks(rotation=45)
  plt.xlabel("Model")
  plt.ylabel("Macro-F1")
  plt.tight_layout()
  plt.savefig(save_path, dpi=300)
  plt.close()


def plot_accuracy_vs_f1(df, save_path):
  sns.set(style="whitegrid")
  plt.figure(figsize=(8, 6))
  sns.scatterplot(data=df, x="accuracy_mean", y="f1_mean", hue="model", style="mode")
  plt.xlabel("Accuracy")
  plt.ylabel("Macro-F1")
  plt.tight_layout()
  plt.savefig(save_path, dpi=300)
  plt.close()


def plot_size_vs_f1(df, save_path):
  sns.set(style="whitegrid")
  order = ["tiny", "small", "base"]

  df["size"] = pd.Categorical(df["size"], categories=order, ordered=True)
  df = df.sort_values("size")

  plt.figure(figsize=(8, 6))
  sns.lineplot(data=df, x="size", y="f1_mean", hue="model", style="encoder", markers=True, dashes=False, sort=False, errorbar=None)

  plt.xticks(range(len(order)), order)
  plt.xlabel("Model Size")
  plt.ylabel("Macro-F1")
  plt.tight_layout()
  plt.savefig(save_path, dpi=300)
  plt.close()


def plot_freeze_comparison(df, save_path):
  sns.set(style="whitegrid")
  plt.figure(figsize=(8, 6))
  sns.barplot(data=df, x="model", y="f1_mean", hue="encoder")
  plt.xticks(rotation=45)
  plt.xlabel("Model")
  plt.ylabel("Macro-F1")
  plt.tight_layout()
  plt.savefig(save_path, dpi=300)
  plt.close()


def plot_delta_from_best(df, save_path):
  sns.set(style="whitegrid")
  df = df.copy()
  best_per_mode = df.groupby("mode")["f1_mean"].transform("max")
  df["delta"] = best_per_mode - df["f1_mean"]
  plt.figure(figsize=(8, 6))
  sns.barplot(data=df, x="model", y="delta", hue="mode")
  plt.xticks(rotation=45)
  plt.xlabel("Model")
  plt.ylabel("Delta from Best F1")
  plt.tight_layout()
  plt.savefig(save_path, dpi=300)
  plt.close()


# model efficiency table (shows model, approximate params, and Macro-F1)
def generate_efficiency_table(df):
  param_map = {
      "rf": 1e5,
      "mlp": 1e5,
      "cnnlstm": 1e6,
      "mae": 1e6,
      "stmae": 1e6,
      "distilbert": 6.6e7,
      "tinyllama": 1.1e9
  }

  df = df.copy()
  df["params"] = df["model"].map(param_map)

  table = (
      df.groupby("model").agg(params=("params", "first"), f1=("f1_mean", "max")).reset_index()
  )

  table.to_csv(os.path.join(RESULTS_ROOT, "model_efficiency_table.csv"), index=False)


# main experimental loop
def main():
  num_failures = 0

  summary_rows = []

  for repeat in range(TARGET_REPEAT, REPEATS):
    for model_id, model_name in MODEL_REGISTRY.items():
      if model_name not in TARGET_MODELS:
        continue

      allowed_modes = MODEL_MODE_MAP.get(model_name, [])
      if not allowed_modes:
        print(f"No allowed modes configured for {model_name}, skipping")
        continue

      for mode in allowed_modes:
        mode_name = MODE_NAMES[mode]
        metrics_dir, plots_dir = create_result_dirs(mode_name)

        for size in MODEL_SIZES[model_name]:
          if num_failures > 3:
            raise Exception("Too many failures check code")

          results = None

          if model_name in ["rf", "mlp", "cnnlstm"]:
            tag = f"{model_name}_{mode_name}_{size}_run_{repeat}"
            freeze = None
            freeze_key = "none"

            if model_name == "cnnlstm":
              if is_completed(model_name, size, mode_name, freeze_key, repeat):
                print(f"Skipping {tag}")
                continue

              else:
                print(f"Running {tag}")
                try:
                  results = run_experiment(model_id, mode, freeze, size, window=9)
                  save_metrics(results, tag, metrics_dir)
                  mark_completed(model_name, size, mode_name, freeze_key, repeat)
                  plot_loss(results['history'], os.path.join(plots_dir, f"{tag}_loss.png"))
                  np.save(os.path.join(plots_dir, f"{tag}_cm.npy"), results['confusion_matrix'])
                  plot_confusion(results['confusion_matrix'], os.path.join(plots_dir, f"{tag}_cm.png"))
                  summary_rows.append({
                    "model": model_name,
                    "size": size,
                    "mode": mode_name,
                    "encoder": "none" if freeze is None else ("frozen" if freeze else "unfrozen"),
                    "accuracy": float(results['accuracy']),
                    "precision": float(results['precision']),
                    "recall": float(results['recall']),
                    "f1": float(results['f1'])
                  })

                except Exception as e:
                  print(f"Error running {tag}: {e}")
                  num_failures += 1
                  continue

                finally:
                  if "results" in locals() and results is not None:
                    del results

                  if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                  gc.collect()
                  print(f"Finished {tag}")

            elif model_name == "rf" or model_name == "mlp":
              if is_completed(model_name, size, mode_name, freeze_key, repeat):
                print(f"Skipping {tag}")
                continue

              else:
                print(f"Running {tag}")
                try:
                  results = run_experiment(model_id, mode, freeze, size)
                  save_metrics(results, tag, metrics_dir)
                  mark_completed(model_name, size, mode_name, freeze_key, repeat)
                  plot_loss(results['history'], os.path.join(plots_dir, f"{tag}_loss.png"))
                  np.save(os.path.join(plots_dir, f"{tag}_cm.npy"), results['confusion_matrix'])
                  plot_confusion(results['confusion_matrix'], os.path.join(plots_dir, f"{tag}_cm.png"))
                  summary_rows.append({
                    "model": model_name,
                    "size": size,
                    "mode": mode_name,
                    "encoder": "none" if freeze is None else ("frozen" if freeze else "unfrozen"),
                    "accuracy": float(results['accuracy']),
                    "precision": float(results['precision']),
                    "recall": float(results['recall']),
                    "f1": float(results['f1'])
                  })

                except Exception as e:
                  print(f"Error running {tag}: {e}")
                  num_failures += 1
                  continue

                finally:
                  if "results" in locals() and results is not None:
                    del results

                  if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                  gc.collect()
                  print(f"Finished {tag}")

          elif model_name in ["mae", "stmae", "tinyllama", "distilbert"]:
            for freeze in FREEZE_OPTIONS[model_name]:
              tag = (
                  f"{model_name}_{mode_name}_{size}_{'frozen' if freeze else 'unfrozen'}_run_{repeat}"
              )
              freeze_key = "frozen" if freeze else "unfrozen"

              if model_name == "mae":
                if is_completed(model_name, size, mode_name, freeze_key, repeat):
                  print(f"Skipping {tag}")
                  continue

                else:
                  print(f"Running {tag}")
                  try:
                    results = run_experiment(model_id, mode, freeze, size)
                    save_metrics(results, tag, metrics_dir)
                    mark_completed(model_name, size, mode_name, freeze_key, repeat)
                    plot_loss(results['history'], os.path.join(plots_dir, f"{tag}_loss.png"))
                    np.save(os.path.join(plots_dir, f"{tag}_cm.npy"), results['confusion_matrix'])
                    plot_confusion(results['confusion_matrix'], os.path.join(plots_dir, f"{tag}_cm.png"))
                    summary_rows.append({
                      "model": model_name,
                      "size": size,
                      "mode": mode_name,
                      "encoder": "none" if freeze is None else ("frozen" if freeze else "unfrozen"),
                      "accuracy": float(results['accuracy']),
                      "precision": float(results['precision']),
                      "recall": float(results['recall']),
                      "f1": float(results['f1'])
                    })

                  except Exception as e:
                    print(f"Error running {tag}: {e}")
                    num_failures += 1
                    continue

                  finally:
                    if "results" in locals() and results is not None:
                      del results

                    if torch.cuda.is_available():
                      torch.cuda.empty_cache()
                    gc.collect()
                    print(f"Finished {tag}")

              elif model_name == "stmae":
                if is_completed(model_name, size, mode_name, freeze_key, repeat):
                  print(f"Skipping {tag}")
                  continue

                else:
                  print(f"Running {tag}")
                  try:
                    results = run_experiment(model_id, mode, freeze, size, window=32)
                    save_metrics(results, tag, metrics_dir)
                    mark_completed(model_name, size, mode_name, freeze_key, repeat)
                    plot_loss(results['history'], os.path.join(plots_dir, f"{tag}_loss.png"))
                    np.save(os.path.join(plots_dir, f"{tag}_cm.npy"), results['confusion_matrix'])
                    plot_confusion(results['confusion_matrix'], os.path.join(plots_dir, f"{tag}_cm.png"))
                    summary_rows.append({
                      "model": model_name,
                      "size": size,
                      "mode": mode_name,
                      "encoder": "none" if freeze is None else ("frozen" if freeze else "unfrozen"),
                      "accuracy": float(results['accuracy']),
                      "precision": float(results['precision']),
                      "recall": float(results['recall']),
                      "f1": float(results['f1'])
                    })

                  except Exception as e:
                    print(f"Error running {tag}: {e}")
                    num_failures += 1
                    continue

                  finally:
                    if "results" in locals() and results is not None:
                      del results

                    if torch.cuda.is_available():
                      torch.cuda.empty_cache()
                    gc.collect()
                    print(f"Finished {tag}")

              elif model_name == "tinyllama":
                if should_skip(model_name, size, freeze):
                  print(f"Skipping redundant run: {model_name}_{size}_{freeze}")
                  continue

                elif is_completed(model_name, size, mode_name, freeze_key, repeat):
                  print(f"Skipping {tag}")
                  continue

                else:
                  print(f"Running {tag}")
                  try:
                    results = run_experiment(model_id, mode, freeze, size, window=3)
                    save_metrics(results, tag, metrics_dir)
                    mark_completed(model_name, size, mode_name, freeze_key, repeat)
                    plot_loss(results['history'], os.path.join(plots_dir, f"{tag}_loss.png"))
                    np.save(os.path.join(plots_dir, f"{tag}_cm.npy"), results['confusion_matrix'])
                    plot_confusion(results['confusion_matrix'], os.path.join(plots_dir, f"{tag}_cm.png"))
                    summary_rows.append({
                      "model": model_name,
                      "size": size,
                      "mode": mode_name,
                      "encoder": "none" if freeze is None else ("frozen" if freeze else "unfrozen"),
                      "accuracy": float(results['accuracy']),
                      "precision": float(results['precision']),
                      "recall": float(results['recall']),
                      "f1": float(results['f1'])
                    })
                    plot_world_model_consistency(
                      results['test_logits'],
                      results['test_labels'],
                      os.path.join(plots_dir, f"{tag}_worldmodel.png")
                    )

                  except Exception as e:
                    print(f"Error running {tag}: {e}")
                    num_failures += 1
                    continue

                  finally:
                    if "results" in locals() and results is not None:
                      del results

                    if torch.cuda.is_available():
                      torch.cuda.empty_cache()
                    gc.collect()
                    print(f"Finished {tag}")

              elif model_name == "distilbert":
                if should_skip(model_name, size, freeze):
                  print(f"Skipping redundant run: {model_name}_{size}_{freeze}")
                  continue

                elif is_completed(model_name, size, mode_name, freeze_key, repeat):
                  print(f"Skipping {tag}")
                  continue

                else:
                  print(f"Running {tag}")
                  try:
                    results = run_experiment(model_id, mode, freeze, size)
                    save_metrics(results, tag, metrics_dir)
                    mark_completed(model_name, size, mode_name, freeze_key, repeat)
                    plot_loss(results['history'], os.path.join(plots_dir, f"{tag}_loss.png"))
                    np.save(os.path.join(plots_dir, f"{tag}_cm.npy"), results['confusion_matrix'])
                    plot_confusion(results['confusion_matrix'], os.path.join(plots_dir, f"{tag}_cm.png"))
                    summary_rows.append({
                      "model": model_name,
                      "size": size,
                      "mode": mode_name,
                      "encoder": "none" if freeze is None else ("frozen" if freeze else "unfrozen"),
                      "accuracy": float(results['accuracy']),
                      "precision": float(results['precision']),
                      "recall": float(results['recall']),
                      "f1": float(results['f1'])
                    })

                  except Exception as e:
                    print(f"Error running {tag}: {e}")
                    num_failures += 1
                    continue


                  finally:
                    if "results" in locals() and results is not None:
                      del results

                    if torch.cuda.is_available():
                      torch.cuda.empty_cache()
                    gc.collect()
                    print(f"Finished {tag}")

          else:
            freeze = None
            freeze_key = "none"
            tag = f"{model_name}_{mode_name}_{size}_run_{repeat}"
            if is_completed(model_name, size, mode_name, freeze_key, repeat):
              print(f"Skipping {tag}")
              continue

            else:
              print(f"Running {tag}")
              try:
                results = run_experiment(model_id, mode, freeze, size)
                save_metrics(results, tag, metrics_dir)
                mark_completed(model_name, size, mode_name, freeze_key, repeat)
                plot_loss(results['history'], os.path.join(plots_dir, f"{tag}_loss.png"))
                np.save(os.path.join(plots_dir, f"{tag}_cm.npy"), results['confusion_matrix'])
                plot_confusion(results['confusion_matrix'], os.path.join(plots_dir, f"{tag}_cm.png"))
                summary_rows.append({
                  "model": model_name,
                  "size": size,
                  "mode": mode_name,
                  "encoder": "none" if freeze is None else ("frozen" if freeze else "unfrozen"),
                  "accuracy": float(results['accuracy']),
                  "precision": float(results['precision']),
                  "recall": float(results['recall']),
                  "f1": float(results['f1'])
                })

              except Exception as e:
                print(f"Error running {tag}: {e}")
                num_failures += 1
                continue

              finally:
                if "results" in locals() and results is not None:
                  del results

                if torch.cuda.is_available():
                  torch.cuda.empty_cache()
                gc.collect()
                print(f"Finished {tag}")

  summary_df = rebuild_summary_from_files()

  summary_df = summary_df.dropna(subset=["accuracy", "precision", "recall", "f1"])

  summary_df = summary_df[~((summary_df["model"].isin(["distilbert", "tinyllama"])) & (summary_df["size"] == "small"))]

  agg_df = (
      summary_df
      .groupby(["model", "family", "size", "mode", "encoder"], as_index=False)
      .agg(
          accuracy_mean=("accuracy", "mean"),
          accuracy_std=("accuracy", "std"),
          f1_mean=("f1", "mean"),
          f1_std=("f1", "std"),
      )
  )

  agg_df.to_csv(os.path.join(RESULTS_ROOT, "summary_clean.csv"), index=False)

  generate_efficiency_table(agg_df)

  plots_dir = os.path.join(RESULTS_ROOT, "summary_plots")
  os.makedirs(plots_dir, exist_ok=True)

  CM_MODEL_MODE_MAP = {
      "rf": ("network", "base", "none"),
      "cnnlstm": ("network", "base", "none"),
      "mae": ("fusion", "base", "unfrozen"),
      "tinyllama": ("fusion", "base", "unfrozen")
  }

  for model, (mode_name, size, encoder) in CM_MODEL_MODE_MAP.items():
    matrix = load_cm_from_results(model, mode_name, size=size, encoder=encoder)
    if matrix is None:
      print(f"No {mode_name} cm for {model}, skipping")
      continue

    cm_small = simplify_confusion(matrix)
    plot_normalized_confusion(cm_small, os.path.join(plots_dir, f"{model}_{mode_name}_{size}_{encoder}_cm_small.png"))

  plot_f1_vs_model(agg_df, os.path.join(plots_dir, "f1_vs_model.png"))

  plot_accuracy_vs_f1(agg_df, os.path.join(plots_dir, "accuracy_vs_f1.png"))

  df_unfrozen = agg_df[agg_df["encoder"] == "unfrozen"]
  df_unfrozen = df_unfrozen.dropna(subset=["f1_mean"])
  df_baseline = agg_df[agg_df["encoder"].isin(["frozen", "none"])]
  plot_size_vs_f1(df_unfrozen, os.path.join(plots_dir, "size_vs_f1_encoder.png"))
  plot_size_vs_f1(df_baseline, os.path.join(plots_dir, "size_vs_f1_no_encoder.png"))

  plot_freeze_comparison(agg_df, os.path.join(plots_dir, "freeze_vs_unfreeze.png"))

  plot_delta_from_best(agg_df, os.path.join(plots_dir, "delta_from_best.png"))



if __name__ == "__main__":
  main()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_2248/1237027849.py:385: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["size"] = pd.Categorical(df["size"], categories=order, ordered=True)
